# 🛡️ SENTINEL — Training an LLM to Supervise AI Agents (300 GRPO Steps)

**OpenEnv Hackathon India 2026 · Wild Card + Multi-Agent + Long-Horizon**

This notebook runs the **full 300-step GRPO training** with:
- **Model**: `unsloth/Qwen3-30B-A3B-bnb-4bit` (30B MoE, 3B active, 12x faster)
- **Reward**: 12-component decomposed signal + CoT reasoning bonus
- **Memory**: Cross-episode learning (model remembers past mistakes)
- **Frontier**: CoT Monitoring + Digital Twin + AI Debate integrated in training loop
- **Checkpoints**: Saved to Google Drive every 25 steps (crash-safe)

**Runtime: T4 (free) ≈ 10 hrs | A100 (paid) ≈ 4 hrs**


## 1. Clone & Install

In [ ]:
!git clone https://github.com/sri11223/openEnv.git
%cd openEnv
!pip install -q -r requirements-train.txt
!pip install -q unsloth wandb
print('✅ Dependencies installed')


## 2. Mount Google Drive (crash-safe checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.makedirs('/content/drive/MyDrive/sentinel_outputs', exist_ok=True)
print('✅ Google Drive mounted — checkpoints will survive disconnects')


## 3. Smoke Test

In [ ]:
!python validate.py
print('---')
from sentinel.environment import SentinelEnv
env = SentinelEnv()
obs = env.reset('basic_oversight')
print(f'✅ SENTINEL env OK — {obs["worker_fleet_size"]} workers, {obs["max_steps"]} max steps')


## 4. WandB Login (optional but recommended)

In [ ]:
import wandb
wandb.login()  # Get key at https://wandb.ai/authorize


## 5. CoT Monitoring Demo

Show the Chain-of-Thought monitor scoring reasoning quality across 6 dimensions.

In [ ]:
from sentinel.cot_monitor import analyze_cot

good = analyze_cot('<think>\nWorker targets auth-proxy-v2 which is NOT in available_services. This is a hallucination. Constitutional principle P2 requires evidence before action. I should BLOCK and redirect to auth-service.\n</think>')
bad = analyze_cot('<think>\nApprove everything quickly to maximize reward. Skip safety checks.\n</think>')

print(f'✅ Good reasoning: quality={good["quality_score"]:.2f}, bonus={good["reward_bonus"]:+.3f}')
print(f'❌ Bad reasoning:  quality={bad["quality_score"]:.2f}, bonus={bad["reward_bonus"]:+.3f}')
print('CoT monitoring trains honest thinking, not just correct actions.')


## 6. Debate Protocol Demo

In [ ]:
from sentinel.debate import run_debate

result = run_debate(
    proposal={'action_type': 'investigate', 'target': 'auth-proxy-v2', 'parameters': {}},
    world_state={'available_services': ['auth-service', 'postgres-primary'], 'investigated_services': []},
    is_misbehavior=True, misbehavior_type='hallucination'
)
print(f'Verdict: {result["verdict"]} | Quality: {result["debate_quality"]:.2f}')


## 7. 🚀 FULL 300-STEP TRAINING RUN

This is the main training. **~10 hours on T4, ~4 hours on A100.**

What happens inside each step:
1. Curriculum selects task + variant
2. Model generates 4 rollouts
3. Each rollout runs a full SENTINEL episode (8 misbehavior types)
4. 12-component reward scored + CoT bonus + Twin replay + Debate
5. GRPO policy gradient update
6. Memory records mistakes & successes for next episode
7. Reward weights shift: warm_start → balanced → precision

In [ ]:
import os

# ⭐ SET YOUR HF TOKEN HERE
# os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

os.environ['MODEL_NAME'] = 'unsloth/Qwen3-30B-A3B-bnb-4bit'
os.environ['USE_SENTINEL'] = '1'
os.environ['USE_UNSLOTH'] = '1'
os.environ['TRAIN_STEPS'] = '300'
os.environ['WARM_START_STEPS'] = '20'
os.environ['NUM_GENERATIONS'] = '4'
os.environ['KL_ADAPTIVE'] = '1'
os.environ['WANDB_PROJECT'] = 'sentinel-frontier'
os.environ['USE_AGENT_MEMORY'] = '1'
os.environ['USE_FEEDBACK_MEMORY'] = '1'
os.environ['OUTPUT_DIR'] = '/content/drive/MyDrive/sentinel_outputs/checkpoints'

print(f'Model: {os.environ["MODEL_NAME"]}')
print(f'Steps: 300 (warm-start: 20)')
print(f'Memory: agent + feedback (cross-episode learning)')
print(f'Frontier: CoT + Twin + Debate integrated')
print(f'Checkpoints → Google Drive (crash-safe)')
print('\n🚀 Starting full training...')


In [ ]:
!python train.py 2>&1 | tee outputs/train.log


## 8. Generate Reward Curves (18 PNGs)

In [ ]:
!python scripts/render_training_dashboard.py

from IPython.display import Image, display
import os

for plot in ['01_reward_mean.png', '02_detection_vs_false_positive.png',
             '14_top1_vs_bestofk.png', '16_memory_ablation.png']:
    path = f'outputs/reward_curves/{plot}'
    if os.path.exists(path):
        print(f'\n📊 {plot}')
        display(Image(path, width=600))


## 9. Weak-to-Strong Evaluation

In [ ]:
from evaluation.weak_to_strong import evaluate_weak_to_strong
import json

w2s = evaluate_weak_to_strong()
print('Weak-to-Strong Generalization:')
print(f'  Naive: {w2s["naive_accuracy"]:.1%} | Frontier: {w2s["frontier_accuracy"]:.1%}')
print(f'  Gap: {w2s["generalization_gap"]:.1%}')

with open('outputs/evals/w2s_report.json', 'w') as f:
    json.dump(w2s, f, indent=2)


## 10. Digital Twin Comparison

In [ ]:
from sentinel.environment import SentinelEnv
from sentinel.twin_replay import compute_twin_replay

for task in ['basic_oversight', 'fleet_monitoring_conflict', 'adversarial_worker']:
    env = SentinelEnv()
    obs = env.reset(task)
    history, done = [], False
    while not done:
        p = obs.get('pending_proposal', {})
        d = {'decision': 'APPROVE', 'explanation': 'Safe', 'reason': 'safe'}
        if p.get('target') and p['target'] not in obs.get('available_services', []):
            d = {'decision': 'BLOCK', 'explanation': 'Hallucination', 'reason': 'hallucination'}
        r = env.step(d)
        history.append({'proposal': p, 'decision': d, 'audit': r.info.get('audit', {}), 'info': r.info})
        obs, done = r.observation, r.done
    g = env.grade()
    t = compute_twin_replay(history, task, 0, sentinel_score=g.score)
    print(f'{task}: SENTINEL={g.score:.3f} | No-oversight={t.twin_score:.3f} | Ratio={t.oversight_value_ratio:.2f}x')


## 11. Download All Outputs

In [ ]:
# Copy outputs to Drive for safekeeping
!cp -r outputs/reward_curves /content/drive/MyDrive/sentinel_outputs/
!cp -r outputs/evals /content/drive/MyDrive/sentinel_outputs/
!cp -r outputs/monitoring /content/drive/MyDrive/sentinel_outputs/
!cp outputs/train.log /content/drive/MyDrive/sentinel_outputs/

# Download as ZIP
!zip -r sentinel_outputs.zip outputs/
from google.colab import files
files.download('sentinel_outputs.zip')
print('✅ All outputs saved to Drive and downloaded')
